# Thực nghiệm minh họa các phương pháp số chuyên biệt cho bài toán tối ưu phi tuyến 

Notebook này tạo các hình minh họa cho báo cáo bài tập lớn môn Phương pháp số cho học máy. Mã chỉ dùng NumPy và Matplotlib, không dùng thư viện tối ưu chuyên dụng.

Các hình sẽ được lưu vào `image/experiments/`.

In [ ]:
from pathlib import Path
import numpy as np

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Notebook cần matplotlib để vẽ hình. Hãy cài bằng: pip install matplotlib") from exc

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIG_DIR = PROJECT_ROOT / "image" / "experiments"
FIG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

def savefig(name: str) -> None:
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Đã lưu: {path}")

## 1. Gauss-Newton và Levenberg-Marquardt cho bài toán khớp đường cong

Ta khớp mô hình $y = c e^{at}$ từ dữ liệu nhiễu. Gauss-Newton dùng bài toán bình phương tối thiểu tuyến tính hóa ở mỗi vòng lặp; Levenberg-Marquardt thêm tham số giảm chấn để ổn định hơn khi khởi tạo xa.

In [ ]:
def exp_residual_and_jac(theta, t, y):
    a, c = theta
    exp_at = np.exp(a * t)
    residual = y - c * exp_at
    jac = np.column_stack((-c * t * exp_at, -exp_at))
    return residual, jac


def nls_objective(theta, t, y):
    residual, _ = exp_residual_and_jac(theta, t, y)
    return 0.5 * float(residual @ residual)


def gauss_newton(theta0, t, y, max_iter=80, tol_step=1e-10, tol_obj=1e-12):
    theta = np.array(theta0, dtype=float)
    history = []
    path = [theta.copy()]

    for _ in range(max_iter):
        residual, jac = exp_residual_and_jac(theta, t, y)
        obj = 0.5 * float(residual @ residual)
        history.append(obj)

        delta, *_ = np.linalg.lstsq(jac, -residual, rcond=None)
        theta_new = theta + delta
        path.append(theta_new.copy())

        if np.linalg.norm(delta) < tol_step:
            theta = theta_new
            break

        new_obj = nls_objective(theta_new, t, y)
        if abs(obj - new_obj) <= tol_obj * max(1.0, obj):
            theta = theta_new
            break

        theta = theta_new

    history.append(nls_objective(theta, t, y))
    return theta, np.array(history), np.array(path)


def levenberg_marquardt(
    theta0,
    t,
    y,
    max_iter=40,
    lam0=1.0,
    tol_step=1e-9,
    tol_obj=1e-10,
    max_reject=12,
):
    theta = np.array(theta0, dtype=float)
    lam = float(lam0)
    history = []
    path = [theta.copy()]

    for _ in range(max_iter):
        residual, jac = exp_residual_and_jac(theta, t, y)
        obj = 0.5 * float(residual @ residual)
        history.append(obj)

        accepted = False
        for _ in range(max_reject):
            lhs = jac.T @ jac + lam * np.eye(2)
            rhs = -jac.T @ residual
            delta = np.linalg.solve(lhs, rhs)
            candidate = theta + delta
            candidate_obj = nls_objective(candidate, t, y)

            if candidate_obj < obj:
                theta = candidate
                path.append(theta.copy())
                lam = max(lam * 0.5, 1e-12)
                accepted = True
                break

            lam = min(lam * 5.0, 1e12)

        if not accepted:
            break

        if np.linalg.norm(delta) < tol_step:
            break

        if abs(obj - candidate_obj) <= tol_obj * max(1.0, obj):
            break

    history.append(nls_objective(theta, t, y))
    return theta, np.array(history), np.array(path)


t = np.linspace(0.0, 3.0, 60)
a_true, c_true = 0.75, 1.4
y_clean = c_true * np.exp(a_true * t)
y = y_clean + rng.normal(scale=0.18, size=t.size)
grid_t = np.linspace(t.min(), t.max(), 250)

initial_points = {
    "near": np.array([0.45, 1.0]),
    "far": np.array([-0.35, 0.5]),
}

results = {}
for name, theta0 in initial_points.items():
    theta_gn, hist_gn, path_gn = gauss_newton(theta0, t, y)
    theta_lm, hist_lm, path_lm = levenberg_marquardt(theta0, t, y)
    results[name] = {
        "theta0": theta0,
        "gn": {"theta": theta_gn, "history": hist_gn, "path": path_gn},
        "lm": {"theta": theta_lm, "history": hist_lm, "path": path_lm},
    }

    print(f"Initialization {name}: theta0={theta0}")
    print(f"  GN cuối: theta={theta_gn}, hàm mục tiêu={hist_gn[-1]:.6g}, số vòng lặp={len(hist_gn) - 1}")
    print(f"  LM cuối: theta={theta_lm}, hàm mục tiêu={hist_lm[-1]:.6g}, số vòng lặp={len(hist_lm) - 1}")

# Figure 1: near initialization. Both GN and LM should converge well.
near = results["near"]
plt.figure()
plt.scatter(t, y, s=22, alpha=0.75, label="Dữ liệu nhiễu")
plt.plot(grid_t, c_true * np.exp(a_true * grid_t), "k--", label="Mô hình thật")
for method, label in [("gn", "GN"), ("lm", "LM")]:
    theta = near[method]["theta"]
    plt.plot(grid_t, theta[1] * np.exp(theta[0] * grid_t), label=f"{label}: a={theta[0]:.3f}, c={theta[1]:.3f}")
plt.xlabel("t")
plt.ylabel("y")
plt.title("Khớp đường cong từ khởi tạo gần")
plt.legend()
savefig("curve_fit_gn_lm_near.png")
plt.show()

# Figure 2: far initialization. GN can move toward a degenerate solution.
far = results["far"]
plt.figure()
plt.scatter(t, y, s=22, alpha=0.75, label="Dữ liệu nhiễu")
plt.plot(grid_t, c_true * np.exp(a_true * grid_t), "k--", label="Mô hình thật")
for method, label in [("gn", "GN"), ("lm", "LM")]:
    theta = far[method]["theta"]
    plt.plot(grid_t, theta[1] * np.exp(theta[0] * grid_t), label=f"{label}: a={theta[0]:.3f}, c={theta[1]:.3g}")
plt.xlabel("t")
plt.ylabel("y")
plt.title("Khớp đường cong từ cùng khởi tạo xa")
plt.legend()
savefig("curve_fit_gn_lm_far.png")
plt.show()

# Hình 3: lịch sử hàm mục tiêu cho hai cách khởi tạo.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=False)
for ax, name, title in zip(axes, ["near", "far"], ["Kh\u1edfi t\u1ea1o g\u1ea7n", "Kh\u1edfi t\u1ea1o xa"]):
    run = results[name]
    ax.semilogy(run["gn"]["history"], marker="o", label="Gauss-Newton")
    ax.semilogy(run["lm"]["history"], marker="s", label="Levenberg-Marquardt")
    ax.set_xlabel("Vòng lặp")
    ax.set_title(title)
    ax.legend()
for ax in axes:
    ax.set_ylabel("Hàm mục tiêu NLS")
fig.suptitle("So sánh GN và LM từ cùng điểm khởi tạo")
fig.tight_layout()
savefig("convergence_gn_lm_near_far.png")
plt.show()


## 2. IRLS cho hồi quy L1 khi có ngoại lai

Bình phương tối thiểu bị kéo bởi ngoại lai vì phạt bình phương sai số dư. IRLS xấp xỉ hồi quy L1 bằng một chuỗi bài toán bình phương tối thiểu có trọng số.

In [ ]:
def irls_l1(A, b, max_iter=60, delta=1e-3):
    x = np.linalg.lstsq(A, b, rcond=None)[0]
    history = []
    for _ in range(max_iter):
        residual = A @ x - b
        history.append(float(np.sum(np.abs(residual))))
        weights = 1.0 / np.maximum(np.abs(residual), delta)
        Aw = A * np.sqrt(weights[:, None])
        bw = b * np.sqrt(weights)
        x_new = np.linalg.lstsq(Aw, bw, rcond=None)[0]
        if np.linalg.norm(x_new - x) < 1e-10:
            x = x_new
            break
        x = x_new
    history.append(float(np.sum(np.abs(A @ x - b))))
    return x, np.array(history)

n = 80
x_data = np.linspace(-3.0, 3.0, n)
A = np.column_stack((np.ones(n), x_data))
beta_true = np.array([1.0, 2.0])
b = A @ beta_true + rng.normal(scale=0.35, size=n)
outlier_idx = rng.choice(n, size=8, replace=False)
b[outlier_idx] += rng.normal(loc=0.0, scale=7.0, size=outlier_idx.size)

beta_l2 = np.linalg.lstsq(A, b, rcond=None)[0]
beta_l1, hist_l1 = irls_l1(A, b)

plt.figure()
plt.scatter(x_data, b, s=24, alpha=0.75, label="Dữ liệu")
plt.scatter(x_data[outlier_idx], b[outlier_idx], s=55, facecolors="none", edgecolors="r", label="Ngoại lai")
plt.plot(x_data, A @ beta_true, "k--", label="Đường thật")
plt.plot(x_data, A @ beta_l2, label="Least-squares")
plt.plot(x_data, A @ beta_l1, label="IRLS L1")
plt.xlabel("x")
plt.ylabel("y")
plt.title("IRLS L1 bền vững hơn trước ngoại lai")
plt.legend()
savefig("irls_l1_regression.png")
plt.show()

plt.figure()
plt.plot(hist_l1, marker="o")
plt.xlabel("Vòng lặp")
plt.ylabel("Hàm mục tiêu L1")
plt.title("Hội tụ IRLS cho hồi quy L1")
savefig("convergence_irls_l1.png")
plt.show()


## 3. Hạ theo tọa độ trong hai chiều

Hình này minh họa quỹ đạo zic-zắc khi lần lượt tối ưu từng tọa độ của một hàm bậc hai lồi.

In [ ]:
Q = np.array([[5.0, 4.0], [4.0, 5.0]])
q = np.array([1.0, -1.2])
z = np.array([-2.5, 2.2])
path = [z.copy()]

for _ in range(18):
    z[0] = (q[0] - Q[0, 1] * z[1]) / Q[0, 0]
    path.append(z.copy())
    z[1] = (q[1] - Q[1, 0] * z[0]) / Q[1, 1]
    path.append(z.copy())

path = np.array(path)
x1 = np.linspace(-3.0, 3.0, 240)
x2 = np.linspace(-3.0, 3.0, 240)
X1, X2 = np.meshgrid(x1, x2)
Z = 0.5 * (Q[0, 0] * X1**2 + 2 * Q[0, 1] * X1 * X2 + Q[1, 1] * X2**2) - q[0] * X1 - q[1] * X2

plt.figure(figsize=(6, 6))
plt.contour(X1, X2, Z, levels=22, cmap="viridis")
plt.plot(path[:, 0], path[:, 1], "o-", color="crimson", markersize=3, label="Hạ theo tọa độ")
optimum = np.linalg.solve(Q, q)
plt.scatter([optimum[0]], [optimum[1]], c="black", s=55, marker="*", label="Nghiệm tối ưu")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Quỹ đạo hạ theo tọa độ")
plt.legend()
plt.axis("equal")
savefig("coordinate_descent_zigzag.png")
plt.show()

## 4. Phân cụm k-means

K-means là tối ưu luân phiên: cố định tâm cụm để gán nhãn, rồi cố định nhãn để cập nhật tâm cụm.

In [ ]:
def kmeans(X, K, max_iter=30, initial_centers=None):
    if initial_centers is None:
        centers = X[rng.choice(X.shape[0], size=K, replace=False)].copy()
    else:
        # Keep later notebook experiments reproducible when this demo uses fixed centers.
        _ = rng.choice(X.shape[0], size=K, replace=False)
        centers = np.array(initial_centers, dtype=float).copy()
    objectives = []
    labels = np.zeros(X.shape[0], dtype=int)
    for _ in range(max_iter):
        distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = np.argmin(distances, axis=1)
        objectives.append(float(np.sum(np.min(distances, axis=1))))
        new_centers = centers.copy()
        for k in range(K):
            members = X[labels == k]
            if len(members) > 0:
                new_centers[k] = members.mean(axis=0)
        if np.allclose(new_centers, centers):
            centers = new_centers
            break
        centers = new_centers
    distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    labels = np.argmin(distances, axis=1)
    objectives.append(float(np.sum(np.min(distances, axis=1))))
    return centers, labels, np.array(objectives)

cluster_means = np.array([[-2.2, -1.2], [2.1, -0.6], [0.0, 2.2]])
X = np.vstack([mu + rng.normal(scale=0.55, size=(70, 2)) for mu in cluster_means])
initial_kmeans_centers = np.array([[-3.0, 3.0], [3.0, 3.0], [0.0, -2.5]])
centers, labels, hist_kmeans = kmeans(X, K=3, initial_centers=initial_kmeans_centers)

plt.figure(figsize=(6, 5))
for k in range(3):
    plt.scatter(X[labels == k, 0], X[labels == k, 1], s=22, alpha=0.75, label=f"Cụm {k + 1}")
plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=140, label="Tâm cụm")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Phân cụm k-means")
plt.legend()
savefig("kmeans_clusters.png")
plt.show()

plt.figure()
plt.plot(hist_kmeans, marker="o")
plt.xlabel("Vòng lặp")
plt.ylabel("Hàm mục tiêu k-means")
plt.title("Hàm mục tiêu của k-means giảm đơn điệu")
savefig("kmeans_objective.png")
plt.show()


## 5. ADMM cho bình phương tối thiểu không âm

Bài toán: $\min_x \|Ax-b\|_2^2$ với ràng buộc $x \ge 0$. ADMM tách thành bước giải hệ tuyến tính, bước chiếu lên miền không âm, và bước cập nhật biến đối ngẫu.

In [ ]:
def admm_nnls(A, b, rho=1.0, max_iter=120):
    m, n = A.shape
    x = np.zeros(n)
    z = np.zeros(n)
    lam = np.zeros(n)
    lhs = 2.0 * A.T @ A + rho * np.eye(n)
    rhs_base = 2.0 * A.T @ b
    obj_hist, pri_hist, dual_hist = [], [], []
    for _ in range(max_iter):
        z_old = z.copy()
        x = np.linalg.solve(lhs, rhs_base + rho * z - lam)
        z = np.maximum(x + lam / rho, 0.0)
        lam = lam + rho * (x - z)
        obj_hist.append(float(np.linalg.norm(A @ x - b) ** 2))
        pri_hist.append(float(np.linalg.norm(x - z)))
        dual_hist.append(float(rho * np.linalg.norm(z - z_old)))
    return x, z, np.array(obj_hist), np.array(pri_hist), np.array(dual_hist)

m, n = 90, 22
A_nnls = rng.normal(size=(m, n))
x_true = np.zeros(n)
active = rng.choice(n, size=7, replace=False)
x_true[active] = rng.uniform(0.4, 2.5, size=active.size)
b_nnls = A_nnls @ x_true + rng.normal(scale=0.05, size=m)

x_admm, z_admm, obj_admm, pri_admm, dual_admm = admm_nnls(A_nnls, b_nnls, rho=1.2)

plt.figure()
plt.semilogy(obj_admm, label="Hàm mục tiêu")
plt.semilogy(pri_admm, label="Phần dư nguyên thủy ||x-z||")
plt.semilogy(dual_admm, label="Phần dư đối ngẫu")
plt.xlabel("Vòng lặp")
plt.title("ADMM cho bình phương tối thiểu không âm")
plt.legend()
savefig("admm_nnls_convergence.png")
plt.show()

plt.figure(figsize=(7, 4))
idx = np.arange(n)
plt.stem(idx - 0.12, x_true, linefmt="k-", markerfmt="ko", basefmt=" ", label="x thật")
plt.stem(idx + 0.12, z_admm, linefmt="C1-", markerfmt="C1o", basefmt=" ", label="ADMM")
plt.xlabel("Chỉ số hệ số")
plt.ylabel("Giá trị")
plt.title("Nghiệm không âm ước lượng bằng ADMM")
plt.legend()
savefig("admm_nnls_solution.png")
plt.show()

## 6. Minh họa hình phạt bậc hai trong Lagrangian tăng cường

Hình này minh họa trực giác: khi tăng hệ số phạt, hàm mục tiêu bị kéo mạnh về miền thỏa ràng buộc $x+y=1$.

In [ ]:
xx = np.linspace(-1.0, 2.0, 220)
yy = np.linspace(-1.0, 2.0, 220)
XX, YY = np.meshgrid(xx, yy)
rhos = [0.0, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(1, len(rhos), figsize=(14, 3.4), sharex=True, sharey=True)
for ax, rho in zip(axes, rhos):
    penalty_obj = XX * YY + 0.5 * rho * (XX + YY - 1.0) ** 2
    levels = np.linspace(np.percentile(penalty_obj, 5), np.percentile(penalty_obj, 95), 18)
    ax.contour(XX, YY, penalty_obj, levels=levels, cmap="viridis")
    ax.plot(xx, 1.0 - xx, "r--", linewidth=1.5, label="x + y = 1")
    ax.set_title(f"rho = {rho}")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper right", fontsize=8)
fig.suptitle("Quadratic penalty kéo nghiệm về tập ràng buộc")
savefig("augmented_lagrangian_penalty.png")
plt.show()

print("Hoàn tất. Các hình đã được lưu trong:", FIG_DIR)